# Protest tactics: stipulated synthetic compatibility computation

Tactic shares, event totals, repression, and the outcome score are stipulated. Nothing below detects strategic interaction, a repression effect, a concession effect, a cycle, a regime, or a motion operator.

The legacy compatibility pipeline is scientifically flawed as a detector, has no validated detection capability, and is not a mixture-motion estimator. A centered-log-ratio motion-estimator repair is designed but is not implemented in this notebook. The computations below expose compatibility behavior only.

NMF is a full-sample reconstruction tool here. The same nonnegative observation matrix is passed to NMF and membership re-inference. Every R2 is in-sample. Equilibrium records describe only the fitted matrix; they are not findings about the data-generating process.


In [ ]:
# Imports and deterministic seed
import os
import random
import numpy as np
import ts2eg as gm

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
TS2EG_CI = os.getenv("TS2EG_CI", "0") == "1"


## Stipulated tactic shares and event counts


In [ ]:
rng = np.random.default_rng(SEED)
T = 80
t = np.arange(T)
repression = np.clip(
    0.6 + 0.18 * np.sin(2 * np.pi * t / 20)
    + 0.05 * rng.standard_normal(T), 0.2, 1.1
)
base = np.array([0.40, 0.28, 0.20, 0.12])[:, None]
variation = np.vstack([
    0.05 * np.sin(2 * np.pi * t / 18),
    0.04 * np.cos(2 * np.pi * (t + 3) / 24),
    0.03 * np.sin(2 * np.pi * (t + 5) / 28),
    0.02 * np.cos(2 * np.pi * (t + 1) / 16),
])
X_model = np.maximum(base + variation + 0.01 * rng.standard_normal((4, T)), 1e-8)
X_model /= X_model.sum(axis=0, keepdims=True)
total_events = 180 * np.exp(-0.5 * (repression - repression.mean()))
counts = rng.poisson(np.maximum(X_model * total_events, 1e-6)) + 1
print("tactic shares:", X_model.shape, "event counts:", counts.shape)


## Growth payoff compatibility computation


In [ ]:
v_growth = gm.growth_payoffs(counts, dt=1.0, pad="edge")
S, H = gm.nmf_on_X(X_model, k=3, iters=120, seed=SEED, normalize="l2")
fit_growth = gm.estimate_A_from_series(S, X_model, v_growth, k=3, ridge=1e-2)
records = gm.find_ESS(fit_growth["A"], max_support=3)
summary = [
    (r["support"], r["ess_status"], r["stability_status"])
    for r in records if r["is_nash"]
]
print("Legacy descriptive growth payoff-field R2 (in-sample):", round(fit_growth["R2"], 3))
print("Fitted-matrix Nash/ESS/stability descriptions:", summary)


## Stipulated same-time outcome association

Rolling LOFO values are in-sample associations, not forecasts of concessions and not causal estimates.


In [ ]:
outcome_score = (
    0.5 * X_model[0] + 0.7 * X_model[1] - 0.5 * X_model[3]
    - 0.2 * repression + 0.04 * rng.standard_normal(T)
)
v_lofo = gm.info_gain_payoffs(X_model, outcome_score, ridge=1e-2, window=20)
fit_lofo = gm.estimate_A_from_series(S, X_model, v_lofo, k=3, ridge=1e-2)
print("Legacy descriptive LOFO payoff-field R2 (in-sample):", round(fit_lofo["R2"], 3))
